# The Ranking System — Which 100 Merchants Should We Onboard?**Buy Now, Pay Later merchant ranking | MAST30034 Applied Data Science**The firm can onboard at most 100 of roughly 4,000 interested merchants thisyear. This notebook explains how we chose them.The question we are answering is deliberately **not** "who is the biggestmerchant?" It is "which hundred partnerships will earn us the most, withoutexposing us to risk we can't price?" Those are different questions, and aranking built on transaction volume answers the wrong one.Run `python scripts/etl_04_ranking.py` before this notebook.

In [ ]:
import sysfrom pathlib import Pathimport matplotlib.pyplot as pltimport numpy as npimport pandas as pdsys.path.insert(0, str(Path.cwd().parent / "scripts"))import configfrom etl_04_ranking import PILLAR_WEIGHTS, RISK_WEIGHTSpd.set_option("display.float_format", lambda v: f"{v:,.2f}")plt.rcParams.update({"figure.figsize": (10, 4), "axes.grid": True, "grid.alpha": 0.3})rankings = pd.read_parquet(config.CURATED_DIR / "merchant_rankings.parquet")established = rankings[rankings["cohort"] == "established"]print(f"merchants scored:        {len(rankings):,}")print(f"established cohort:      {len(established):,}")print(f"insufficient history:    {len(rankings) - len(established):,}")

## 1. How the score is builtThree pillars, each answering a question a partnerships manager would actuallyask about a prospective merchant.| Pillar | Weight | The question it answers ||---|---|---|| **Value** | 0.50 | How much will this merchant earn us next year? || **Growth** | 0.25 | Is that number heading up or down? || **Risk** | 0.25 | How likely is that number to be wrong, or to disappear? |### Why percentiles rather than raw valuesThe underlying features are enormously skewed — the largest merchant processestens of thousands of times more transactions than the smallest. A weighted sumof raw values would be a ranking of transaction volume with three decorativeextra terms, no matter what weights we chose. Converting each pillar to apercentile first puts them on the same 0–1 footing, so the weights mean whatthey say.

In [ ]:
pillars = pd.DataFrame({    "pillar": list(PILLAR_WEIGHTS.keys()),    "weight": list(PILLAR_WEIGHTS.values()),})risk = pd.DataFrame({    "risk component": list(RISK_WEIGHTS.keys()),    "weight within risk": list(RISK_WEIGHTS.values()),})display(pillars)display(risk)

### The headline number`projected_annual_revenue` is the one quantity here that a non-technicalstakeholder can price directly: the dollars we expect this merchant to generatefor the firm over the next twelve months, calculated as their observed monthlyrevenue extended forward and adjusted by their revenue trend.The trend adjustment is clipped at ±30%. Without a clip, a merchant whoserevenue happened to double across a nine-month window projects to an absurdfigure and takes the top of the ranking on noise.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 3.5))for ax, pillar in zip(axes, ["value_score", "growth_score", "risk_score"]):    ax.hist(established[pillar], bins=50, color="#4C72B0")    ax.set(title=pillar.replace("_", " "), xlabel="score (higher is better)")axes[0].set_ylabel("merchants")plt.tight_layout()plt.show()

## 2. Is this just a revenue ranking in disguise?This is the question a sceptical stakeholder should ask, and we should answer ithonestly rather than hope nobody does.

In [ ]:
diagnostics = pd.Series({    "final score vs projected revenue (Spearman)":        established["final_score"].corr(established["projected_annual_revenue"], method="spearman"),    "final score vs transaction count (Spearman)":        established["final_score"].corr(established["n_transactions"], method="spearman"),})display(diagnostics.round(3).to_frame("correlation"))by_score = set(established.nsmallest(100, "overall_rank")["merchant_abn"])by_revenue = set(established.nlargest(100, "projected_annual_revenue")["merchant_abn"])print(f"\nMerchants in both top-100 lists: {len(by_score & by_revenue)}")print(f"In our top 100 but NOT the top 100 by revenue alone: {len(by_score - by_revenue)}")

**The honest answer: it is revenue-dominated, but not revenue alone.**The rank correlation with projected revenue is high, which is expected andarguably correct — the firm is paid a percentage of transaction value, so aranking that ignored revenue would be indefensible. But the overlap between ourtop 100 and a naive top 100 by revenue is only around two thirds. Roughly athird of the list changes once growth and risk are considered.Those swapped-in merchants are where the system earns its keep. Below are themerchants that a pure revenue ranking would have missed.

In [ ]:
swapped_in = established[    established["merchant_abn"].isin(by_score - by_revenue)].nsmallest(10, "overall_rank")swapped_in[[    "overall_rank", "merchant_name", "segment", "projected_annual_revenue",    "growth_score", "risk_score", "final_score",]]

In [ ]:
# And the mirror image: high revenue, but kept out of the top 100.excluded = established[    established["merchant_abn"].isin(by_revenue - by_score)].nlargest(10, "projected_annual_revenue")excluded[[    "overall_rank", "merchant_name", "segment", "projected_annual_revenue",    "revenue_volatility", "customer_hhi", "growth_score", "risk_score",]]

Read the second table alongside the first. The merchants held out of the top 100despite strong revenue are generally there for one of two reasons: their revenueswings sharply month to month, or it rests on a small number of customers. Bothare reasons a partnerships manager would hesitate, and both are exactly what therisk pillar was built to catch.

## 3. The top 100### What they are worth

In [ ]:
top_100 = established.nsmallest(100, "overall_rank")share_of_revenue = (    100 * top_100["projected_annual_revenue"].sum()    / established["projected_annual_revenue"].sum())share_of_merchants = 100 * len(top_100) / len(established)print(f"projected annual revenue from the top 100: "      f"${top_100['projected_annual_revenue'].sum():,.0f}")print(f"that is {share_of_revenue:.1f}% of the established book, "      f"from {share_of_merchants:.1f}% of merchants")

This is the number for the first slide. **Onboarding 3% of interested merchantscaptures roughly 29% of the available revenue.** The constraint the firm isoperating under — only 100 slots — is far less costly than it sounds, providedthe 100 are chosen well.

In [ ]:
ordered = established.sort_values("projected_annual_revenue", ascending=False)cumulative = 100 * ordered["projected_annual_revenue"].cumsum() / ordered["projected_annual_revenue"].sum()fig, ax = plt.subplots()ax.plot(range(1, len(cumulative) + 1), cumulative.values, color="#4C72B0")ax.axvline(100, color="crimson", linestyle="--", label="onboarding limit (100)")ax.set(xlabel="merchants, ranked by projected revenue",       ylabel="cumulative % of projected revenue",       title="Revenue is highly concentrated among a small number of merchants")ax.legend()plt.tight_layout()plt.show()

### Segment compositionThe top 100 is not dominated by any single segment, which is a useful propertyin itself — a partnership book concentrated in one vertical rises and falls withthat vertical.

In [ ]:
composition = pd.DataFrame({    "in top 100": top_100["segment"].value_counts(),    "in full book": established["segment"].value_counts(),})composition["% of segment selected"] = 100 * composition["in top 100"] / composition["in full book"]composition

In [ ]:
top_100[[    "overall_rank", "merchant_name", "segment", "category", "take_rate",    "projected_annual_revenue", "n_customers", "mean_basket", "final_score",]].head(25)

## 4. Top 10 within each segmentRanking within segment matters because segments are not commensurable. Amerchant with a $50 basket and thousands of orders and one with a $3,000 basketand a few hundred are not competing for the same partnership slot, and comparingthem directly favours whichever business model happens to move more dollars.

In [ ]:
segment_leaders = (    established[established["segment_rank"] <= 10]    .sort_values(["segment", "segment_rank"]))for segment in sorted(segment_leaders["segment"].unique()):    print(f"\n=== {segment} ===")    display(        segment_leaders[segment_leaders["segment"] == segment][[            "segment_rank", "merchant_name", "category",            "projected_annual_revenue", "n_customers", "final_score",        ]]    )

## 5. Interesting merchantsThe project brief asks specifically for merchants that are *interesting* ratherthan simply large. Three groups are worth putting in front of a partnershipsmanager.

### 5a. Small customer base, large purchasesThe brief names this case directly. These merchants have few customers but highaverage baskets — jewellers, antique dealers, high-end furniture.

In [ ]:
boutique = established[    (established["n_customers"] < established["n_customers"].quantile(0.25))    & (established["mean_basket"] > established["mean_basket"].quantile(0.90))]print(f"{len(boutique)} merchants match this profile")print(f"median rank in our system: {int(boutique['overall_rank'].median())}")boutique.nsmallest(8, "overall_rank")[[    "overall_rank", "merchant_name", "category", "n_customers",    "mean_basket", "projected_annual_revenue",]]

**These merchants rank poorly, and that is a finding rather than a bug — but itis one worth stating plainly.**Our system is built around expected revenue to the firm, and on that measure aboutique with a hundred customers genuinely is worth less than a mid-sizeretailer with thousands. The ranking is doing what it was asked to do.But it is worth naming what that assumption costs. If the firm's strategy is toattract high-value consumers rather than to maximise near-term take-raterevenue, these merchants are more attractive than their rank suggests, and thevalue pillar would need reweighting toward basket size. That is a decision forthe business, not for us — but the partnerships team should make it knowinglyrather than inherit it from our weights by accident.

### 5b. High growth from a small baseMerchants whose revenue is rising sharply but who are not yet large. These arethe ones worth onboarding before a competitor does.

In [ ]:
risers = established[    (established["revenue_growth_rate"] > established["revenue_growth_rate"].quantile(0.90))    & (established["projected_annual_revenue"] < established["projected_annual_revenue"].quantile(0.60))].nlargest(10, "revenue_growth_rate")risers[[    "overall_rank", "merchant_name", "segment", "revenue_growth_rate",    "projected_annual_revenue", "n_customers", "final_score",]]

### 5c. Concentration riskMerchants whose revenue rests on very few customers. The Herfindahl index belowis computed over each merchant's customer revenue shares: near 0 means revenueis spread widely, near 1 means losing one customer would take most of it.These are not necessarily bad partners, but they should be onboarded with theconcentration understood rather than discovered later.

In [ ]:
concentrated = established[    (established["customer_hhi"] > established["customer_hhi"].quantile(0.95))    & (established["projected_annual_revenue"] > established["projected_annual_revenue"].quantile(0.75))].nlargest(10, "customer_hhi")concentrated[[    "overall_rank", "merchant_name", "segment", "customer_hhi",    "n_customers", "projected_annual_revenue", "risk_score",]]

### 5d. The watchlist — merchants we cannot score yetMerchants below the minimum transaction threshold are held out of the rankingentirely. Scoring them alongside established merchants would let a flatteringnumber built on a handful of transactions outrank demonstrated performance.They are not rejected. They are the "new merchant with little information" casefrom the brief, and the right treatment is to revisit them next cycle with moredata rather than to guess now.

In [ ]:
watchlist = rankings[rankings["cohort"] == "insufficient_history"]print(f"{len(watchlist)} merchants held out of the ranking")print(f"median transactions among them: {watchlist['n_transactions'].median():.0f}")watchlist.nsmallest(10, "watchlist_rank")[[    "watchlist_rank", "merchant_name", "segment", "n_transactions",    "projected_annual_revenue", "final_score",]]

## 6. How sensitive is the top 100 to our weights?The pillar weights are a business judgement. If small changes to them reshuffledthe recommendations entirely, the ranking would not be worth acting on. Thischecks that.

In [ ]:
def score_with(value_w, growth_w, risk_w):    total = value_w + growth_w + risk_w    score = (        value_w * established["value_score"]        + growth_w * established["growth_score"]        + risk_w * established["risk_score"]    ) / total    return set(established.loc[score.nlargest(100).index, "merchant_abn"])baseline = score_with(0.50, 0.25, 0.25)scenarios = {    "baseline (0.50 / 0.25 / 0.25)": (0.50, 0.25, 0.25),    "revenue focused (0.70 / 0.15 / 0.15)": (0.70, 0.15, 0.15),    "risk averse (0.40 / 0.20 / 0.40)": (0.40, 0.20, 0.40),    "growth focused (0.40 / 0.40 / 0.20)": (0.40, 0.40, 0.20),    "equal weights": (1 / 3, 1 / 3, 1 / 3),}sensitivity = pd.DataFrame([    {"scenario": name, "overlap with baseline top 100": len(baseline & score_with(*weights))}    for name, weights in scenarios.items()])sensitivity

Read the overlap column as the stability of the recommendation. A high overlapacross scenarios means the top 100 is driven by merchants who are strong onevery pillar rather than by our specific choice of weights — which is what makesthe list defensible to someone who disagrees with those weights.Where the overlap drops, that scenario is worth discussing with the businessdirectly, since it means the weighting choice is genuinely doing work.

## 7. Limitations1. **No fraud model.** The supplied fraud labels cover a period with no   overlapping transaction data, so the risk pillar uses behavioural proxies —   volatility, concentration, rule-failure rate — rather than modelled fraud.   This is the single largest gap in the current system.2. **Under a year of data.** Growth features are short-run momentum. We cannot   separate a genuine upward trend from seasonality, and should not claim to.3. **No demographic features yet** where the ABS data has not been attached.   Customer income and regional penetration would strengthen the value pillar.4. **The weights are asserted, not learned.** There is no ground truth for   "correct" onboarding decisions, so they cannot be fitted. Section 6 is what   we offer instead: evidence that the recommendation survives reasonable   disagreement about them.5. **Synthetic data.** Every relationship here may be a generator artefact.